# Loan Eligibility Model Training

This notebook creates synthetic lending data, engineers risk ratios, trains an XGBoost classifier, evaluates it, and exports a local `industrial_loan_model.joblib` artifact. It is intended for offline experimentation only; it is not connected to the frontend prototype.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
import joblib

np.random.seed(42)
n_samples = 5000

data = {
    'Age': np.random.randint(21, 65, n_samples),
    'Monthly_Salary': np.random.randint(15000, 150000, n_samples),
    'Requested_Loan_Amount': np.random.randint(50000, 2000000, n_samples),
    'Loan_Tenure_Months': np.random.choice([12, 24, 36, 48, 60], n_samples),
    'Credit_Score': np.random.randint(300, 900, n_samples),
    'Existing_EMI': np.random.randint(0, 40000, n_samples)
}
df = pd.DataFrame(data)

# 1 = eligible / approve, 0 = reject. Noise prevents a deterministic target.
eligibility_score = (df['Credit_Score'] * 0.5) + (df['Monthly_Salary'] / 1000) - (df['Requested_Loan_Amount'] / 10000)
df['Loan_Status'] = np.where(eligibility_score > 350, 1, 0)
noise = np.random.choice([0, 1], size=n_samples, p=[0.9, 0.1])
df['Loan_Status'] = np.abs(df['Loan_Status'] - noise)

print('Data generated. Target distribution:')
print(df['Loan_Status'].value_counts())

In [ ]:
# Feature engineering: affordability and lending leverage ratios.
df['Debt_to_Income_Ratio'] = (df['Requested_Loan_Amount'] / df['Loan_Tenure_Months'] + df['Existing_EMI']) / df['Monthly_Salary']
df['Loan_to_Income_Ratio'] = df['Requested_Loan_Amount'] / (df['Monthly_Salary'] * 12)

features = [
    'Age', 'Monthly_Salary', 'Requested_Loan_Amount', 'Loan_Tenure_Months',
    'Credit_Score', 'Existing_EMI', 'Debt_to_Income_Ratio', 'Loan_to_Income_Ratio'
]
X = df[features]
y = df['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    random_state=42,
    eval_metric='logloss'
)

print('Training XGBoost model…')
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('--- Model evaluation ---')
print(f'Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%')
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')
print(classification_report(y_test, y_pred))

joblib.dump({
    'model': model,
    'features': features,
    'training_note': 'Synthetic-data demonstration model; not for production lending decisions.'
}, 'industrial_loan_model.joblib')
print('Saved industrial_loan_model.joblib')

In [ ]:
# Optional visual explanation of the trained feature importance.
xgb.plot_importance(model)

## Model interpretation

- **Debt-to-income ratio (DTI):** monthly loan obligation plus existing EMI divided by monthly salary. Higher DTI signals greater repayment burden.
- **Credit score:** the model can learn lower repayment likelihood for lower synthetic scores.
- **Gradient boosting:** sequential decision trees learn non-linear combinations of affordability, credit history, income, and tenure.

> Important: This notebook uses synthetic data. It should not be used to make real-world lending, credit, employment, housing, insurance, or other high-impact decisions without appropriate governance, representative data, fairness evaluation, validation, monitoring, and human oversight.